# LeetCode 2069 — Walking Robot Simulation II bằng Python

Link đề: https://leetcode.com/problems/walking-robot-simulation-ii/ · Độ khó: **Medium** · Chủ đề: Design, Simulation

Skeleton LeetCode:

```python
class Robot:

    def __init__(self, width: int, height: int):
        ...

    def step(self, num: int) -> None:
        ...

    def getPos(self) -> List[int]:
        ...

    def getDir(self) -> str:
        ...
```

## 1. Mục tiêu học

1. Nhận ra robot **chỉ đi trên viền** lưới (xuất phát ở góc, chỉ quay khi chạm biên) ⇒ quỹ đạo là một **vòng chu vi** dài `P = 2(w − 1) + 2(h − 1)`.
2. Thu gọn mỗi lệnh `step(num)` thành phép cộng **modulo `P`** trên chỉ số vòng — `O(1)` mỗi lệnh thay vì `O(num)`.
3. Xử lý **trường hợp biên nổi tiếng**: ở ô `(0, 0)` sau khi đã đi (ít nhất một bước), hướng là **South**, không phải East như lúc đầu.
4. Viết **bộ chạy lệnh** và đối chiếu với mô phỏng từng bước trên chuỗi lệnh ngẫu nhiên.

## 2. Đề bài & ví dụ

Lưới `width × height` trên mặt phẳng, ô trái-dưới `(0, 0)`, ô phải-trên `(width − 1, height − 1)`. Robot bắt đầu ở `(0, 0)`, hướng **East**.

Lệnh `step(num)`: lặp `num` lần — thử đi một ô theo hướng hiện tại; nếu ô đó **ra ngoài** lưới, robot **quay 90° ngược chiều kim đồng hồ** rồi thử lại bước đó.

Lớp `Robot`:
- `Robot(width, height)`; `step(num)`; `getPos()` trả `[x, y]`; `getDir()` trả `"North"`, `"East"`, `"South"` hoặc `"West"`.

**Ví dụ 1:**

```
["Robot","step","step","getPos","getDir","step","step","step","getPos","getDir"]
[[6,3],[2],[2],[],[],[2],[1],[4],[],[]]
-> [null,null,null,[4,0],"East",null,null,null,[1,2],"West"]
```

Sau `step(2), step(2)`: `(4, 0)`, East. `step(2)`: tới `(5, 0)`, bước sau ra ngoài ⇒ quay North, tới `(5, 1)`. `step(1)`: `(5, 2)`, North. `step(4)`: North ra ngoài ⇒ quay West, đi 4 ô tới `(1, 2)`.

**Ràng buộc:** `2 ≤ width, height ≤ 100`; `1 ≤ num ≤ 10⁵`; tổng số lời gọi `≤ 10⁴`.

In [1]:
import random
import time
from typing import List

def run(cls, ops, args):
    obj, out = None, []
    for op, a in zip(ops, args):
        if obj is None:
            obj = cls(*a); out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

ex_ops = ["Robot", "step", "step", "getPos", "getDir", "step", "step", "step", "getPos", "getDir"]
ex_args = [[6, 3], [2], [2], [], [], [2], [1], [4], [], []]
ex_out = [None, None, None, [4, 0], "East", None, None, None, [1, 2], "West"]
for op, a, o in zip(ex_ops, ex_args, ex_out):
    print(f"{op:<7} {str(a):<7} -> {o}")

Robot   [6, 3]  -> None
step    [2]     -> None
step    [2]     -> None
getPos  []      -> [4, 0]
getDir  []      -> East
step    [2]     -> None
step    [1]     -> None
step    [4]     -> None
getPos  []      -> [1, 2]
getDir  []      -> West


## 3. Kiến thức nền

### 3.1 Hướng và quay ngược chiều kim đồng hồ

Thứ tự ngược chiều kim đồng hồ: `East → North → West → South → East`. Với vector `(dx, dy)`: East `(1,0)`, North `(0,1)`, West `(−1,0)`, South `(0,−1)`.

In [2]:
NAMES = ["East", "North", "West", "South"]
DV = [(1, 0), (0, 1), (-1, 0), (0, -1)]
d = 0
for _ in range(5):
    print(NAMES[d], DV[d], end=" -> "); d = (d + 1) % 4
print()

East (1, 0) -> North (0, 1) -> West (-1, 0) -> South (0, -1) -> East (1, 0) -> 


### 3.2 Chu vi của viền lưới

Viền `w × h` (với `w, h ≥ 2`) có `2(w − 1) + 2(h − 1)` ô (mỗi góc chỉ tính một lần). Robot đi hết một vòng sau đúng `P` bước.

### 3.3 Phép modulo trên vòng

Vị trí sau `k` bước trên vòng dài `P` là `k mod P`. Nhiều lệnh liên tiếp chỉ cần cộng dồn rồi lấy mod.

### 3.4 Hướng khi **đến** một ô

Robot chỉ quay khi **thử** bước tiếp theo. Vì vậy khi dừng ở một góc, hướng là hướng của cạnh **vừa đi qua**: tới `(w−1, 0)` hướng East, tới `(w−1, h−1)` hướng North, tới `(0, h−1)` hướng West, tới `(0, 0)` (sau một vòng) hướng **South**.

## 4. Ý tưởng nền tảng / trực giác

Đánh số các ô trên viền theo thứ tự robot đi: chỉ số `0` là `(0, 0)`, tăng dần dọc cạnh dưới sang phải, lên cạnh phải, sang trái cạnh trên, xuống cạnh trái và quay về `0`. Trạng thái robot chỉ là **chỉ số `k`** trên vòng (và cờ "đã đi chưa").

Với mỗi chỉ số, hướng "khi đến" được xác định: cạnh dưới (trừ chỉ số 0) East, cạnh phải North, cạnh trên West, cạnh trái South — và chỉ số `0` là South **nếu đã đi**, East **nếu chưa đi**.

Tính tay **Ví dụ 1** (`w = 6, h = 3`, `P = 2·5 + 2·2 = 14`):

| lệnh | tổng bước | k = tổng mod 14 | ô | hướng |
|---|---|---|---|---|
| step 2 | 2 | 2 | (2, 0) | East |
| step 2 | 4 | 4 | (4, 0) | East |
| step 2 | 6 | 6 | (5, 1) | North |
| step 1 | 7 | 7 | (5, 2) | North |
| step 4 | 11 | 11 | (1, 2) | West |

In [3]:
def ring(w, h):
    cells = []
    for x in range(0, w): cells.append(((x, 0), "East"))
    for y in range(1, h): cells.append(((w - 1, y), "North"))
    for x in range(w - 2, -1, -1): cells.append(((x, h - 1), "West"))
    for y in range(h - 2, 0, -1): cells.append(((0, y), "South"))
    cells[0] = ((0, 0), "South")                         # (0,0) khi đã đi ít nhất 1 bước
    return cells

R = ring(6, 3); print("P =", len(R)); print(R)

P = 14
[((0, 0), 'South'), ((1, 0), 'East'), ((2, 0), 'East'), ((3, 0), 'East'), ((4, 0), 'East'), ((5, 0), 'East'), ((5, 1), 'North'), ((5, 2), 'North'), ((4, 2), 'West'), ((3, 2), 'West'), ((2, 2), 'West'), ((1, 2), 'West'), ((0, 2), 'West'), ((0, 1), 'South')]


## 5. Cách 1 — Brute Force / baseline: mô phỏng từng bước đúng như đề

In [4]:
class RobotBrute:
    def __init__(self, width: int, height: int):
        self.w, self.h = width, height
        self.x = self.y = 0
        self.d = 0                                           # East

    def step(self, num: int) -> None:
        for _ in range(num):
            while True:
                nx, ny = self.x + DV[self.d][0], self.y + DV[self.d][1]
                if 0 <= nx < self.w and 0 <= ny < self.h:
                    self.x, self.y = nx, ny; break
                self.d = (self.d + 1) % 4                    # quay ngược chiều kim đồng hồ, thử lại

    def getPos(self) -> List[int]:
        return [self.x, self.y]

    def getDir(self) -> str:
        return NAMES[self.d]

out = run(RobotBrute, ex_ops, ex_args); print(out); assert out == ex_out

[None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


**Độ phức tạp:** `O(num)` mỗi lệnh ⇒ tệ nhất `10⁴ · 10⁵ = 10⁹` bước ⇒ TLE. Dùng làm chuẩn đối chiếu.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (chỉ đi viền):** robot bắt đầu ở góc trên cạnh dưới, chỉ quay khi chạm biên ⇒ không bao giờ rời viền ⇒ trạng thái là chỉ số trên vòng dài `P`.

**Insight 2 (modulo):** `step(num)` ⇒ `k = (k + num) mod P`; đánh dấu `moved = True`.

**Insight 3 (biên `(0,0)`):** sau khi đã đi, hướng tại chỉ số 0 là South (robot tới `(0,0)` từ cạnh trái đi xuống). Chưa đi thì là East.

**Invariant:** `k` = (tổng số bước đã đi) mod `P`; vị trí và hướng của robot bằng `ring[k]` (trừ trường hợp chưa đi).

### 6.1 Minh họa: robot thật sự chỉ ở trên viền

Chạy mô phỏng từng bước nhiều lần ngẫu nhiên và kiểm tra mọi ô đi qua đều nằm trên viền; đồng thời kiểm tra sau đúng `P` bước robot quay lại ô cũ.

In [5]:
rng = random.Random(61)
for _ in range(50):
    w, h = rng.randint(2, 8), rng.randint(2, 8); rb = RobotBrute(w, h); P = 2 * (w - 1) + 2 * (h - 1)
    for _ in range(30):
        rb.step(1); x, y = rb.getPos()
        assert x in (0, w - 1) or y in (0, h - 1)
    before = rb.getPos(); rb.step(P); assert rb.getPos() == before
print("Mọi ô robot đi qua đều trên viền; sau P bước quay về đúng ô cũ")

Mọi ô robot đi qua đều trên viền; sau P bước quay về đúng ô cũ


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Bảng vòng dựng sẵn + chỉ số modulo (bản nộp)

In [6]:
class Robot:
    def __init__(self, width: int, height: int):
        self.cells = []
        for x in range(width):
            self.cells.append((x, 0, "East"))
        for y in range(1, height):
            self.cells.append((width - 1, y, "North"))
        for x in range(width - 2, -1, -1):
            self.cells.append((x, height - 1, "West"))
        for y in range(height - 2, 0, -1):
            self.cells.append((0, y, "South"))
        self.cells[0] = (0, 0, "South")          # (0,0) sau khi đã đi
        self.P = len(self.cells)
        self.k = 0
        self.moved = False

    def step(self, num: int) -> None:
        self.moved = True
        self.k = (self.k + num) % self.P

    def getPos(self) -> List[int]:
        x, y, _ = self.cells[self.k]
        return [x, y]

    def getDir(self) -> str:
        if not self.moved:
            return "East"
        return self.cells[self.k][2]

out = run(Robot, ex_ops, ex_args); print(out); assert out == ex_out

[None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


**Độ phức tạp Cách 2:** khởi tạo `O(w + h)`, mỗi lệnh `O(1)`.

### 7.2 Cách 3 — Tính trực tiếp từ chỉ số (không dựng bảng)

Từ `k`, xác định cạnh bằng các mốc `w − 1`, `w − 1 + h − 1`, `2(w − 1) + h − 1`, `P`.

In [7]:
class RobotFormula:
    def __init__(self, width: int, height: int):
        self.w, self.h = width, height
        self.P = 2 * (width - 1) + 2 * (height - 1)
        self.k = 0; self.moved = False

    def step(self, num: int) -> None:
        self.moved = True; self.k = (self.k + num) % self.P

    def _state(self):
        w, h, k = self.w, self.h, self.k
        if k == 0:
            return 0, 0, ("South" if self.moved else "East")
        if k <= w - 1:
            return k, 0, "East"
        if k <= (w - 1) + (h - 1):
            return w - 1, k - (w - 1), "North"
        if k <= 2 * (w - 1) + (h - 1):
            return (w - 1) - (k - (w - 1) - (h - 1)), h - 1, "West"
        return 0, (h - 1) - (k - 2 * (w - 1) - (h - 1)), "South"

    def getPos(self) -> List[int]:
        x, y, _ = self._state(); return [x, y]

    def getDir(self) -> str:
        return self._state()[2]

print(run(RobotFormula, ex_ops, ex_args))

[None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


### 7.3 Cách 4 — Mô phỏng "theo cạnh" (nhảy cả đoạn) sau khi lấy mod

Lấy `num mod P` (nếu bằng 0 và robot đã đi thì coi như đi đủ một vòng `P`), rồi mô phỏng theo từng **cạnh**: đi tối đa tới góc, quay, đi tiếp. Mỗi lệnh tối đa 4–5 đoạn.

In [8]:
class RobotSegments:
    def __init__(self, width: int, height: int):
        self.w, self.h = width, height; self.x = self.y = 0; self.d = 0
        self.P = 2 * (width - 1) + 2 * (height - 1)

    def step(self, num: int) -> None:
        num %= self.P
        if num == 0:
            num = self.P                              # đi đủ một vòng: vị trí như cũ nhưng hướng có thể đổi
        while num:
            dx, dy = DV[self.d]
            if dx: room = (self.w - 1 - self.x) if dx > 0 else self.x
            else:  room = (self.h - 1 - self.y) if dy > 0 else self.y
            if room == 0:
                self.d = (self.d + 1) % 4; continue
            t = min(room, num)
            self.x += dx * t; self.y += dy * t; num -= t

    def getPos(self) -> List[int]:
        return [self.x, self.y]

    def getDir(self) -> str:
        return NAMES[self.d]

print(run(RobotSegments, ex_ops, ex_args))

[None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


Mẹo `num = P` khi `num % P == 0`: đi đủ một vòng luôn đưa robot về đúng ô cũ, nhưng nếu đang ở `(0, 0)` hướng East (chưa đi) thì sau vòng hướng thành South — mô phỏng một vòng thật xử lý đúng điều đó. Chi phí `O(1)` đoạn mỗi lệnh.

## 8. TRACE từng bước

Trace Cách 2 trên **Ví dụ 1** và một chuỗi lệnh tự dựng chạm biên `(0,0)` (đối chiếu lại đề gốc).

In [9]:
def trace(ops, args):
    rb = None
    for op, a in zip(ops, args):
        if rb is None:
            rb = Robot(*a); print(f"Robot{tuple(a)}: P = {rb.P}"); continue
        r = getattr(rb, op)(*a)
        print(f"{op}{tuple(a)} -> {r}   (k = {rb.k}, moved = {rb.moved})")

trace(ex_ops, ex_args)
print("---")
trace(["Robot", "getDir", "step", "getPos", "getDir", "step", "getDir"], [[3, 2], [], [6], [], [], [1], []])

Robot(6, 3): P = 14
step(2,) -> None   (k = 2, moved = True)
step(2,) -> None   (k = 4, moved = True)
getPos() -> [4, 0]   (k = 4, moved = True)
getDir() -> East   (k = 4, moved = True)
step(2,) -> None   (k = 6, moved = True)
step(1,) -> None   (k = 7, moved = True)
step(4,) -> None   (k = 11, moved = True)
getPos() -> [1, 2]   (k = 11, moved = True)
getDir() -> West   (k = 11, moved = True)
---
Robot(3, 2): P = 6
getDir() -> East   (k = 0, moved = False)
step(6,) -> None   (k = 0, moved = True)
getPos() -> [0, 0]   (k = 0, moved = True)
getDir() -> South   (k = 0, moved = True)
step(1,) -> None   (k = 1, moved = True)
getDir() -> East   (k = 1, moved = True)


| lệnh (lưới 3×2, P = 6) | k | vị trí | hướng |
|---|---|---|---|
| (khởi tạo) | 0 | (0,0) | **East** (chưa đi) |
| step(6) | 0 | (0,0) | **South** (vừa đi xuống cạnh trái) |
| step(1) | 1 | (1,0) | East |

## 9. Code nộp LeetCode

Bản chọn nộp: **Cách 2** — bảng vòng + chỉ số modulo + cờ `moved`.

In [10]:
class Robot:

    def __init__(self, width: int, height: int):
        self.cells = []
        for x in range(width):
            self.cells.append((x, 0, "East"))
        for y in range(1, height):
            self.cells.append((width - 1, y, "North"))
        for x in range(width - 2, -1, -1):
            self.cells.append((x, height - 1, "West"))
        for y in range(height - 2, 0, -1):
            self.cells.append((0, y, "South"))
        self.cells[0] = (0, 0, "South")
        self.P = len(self.cells)
        self.k = 0
        self.moved = False

    def step(self, num: int) -> None:
        self.moved = True
        self.k = (self.k + num) % self.P

    def getPos(self) -> List[int]:
        x, y, _ = self.cells[self.k]
        return [x, y]

    def getDir(self) -> str:
        if not self.moved:
            return "East"
        return self.cells[self.k][2]

out = run(Robot, ex_ops, ex_args); print(out); assert out == ex_out

[None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


## 10. Vì sao thuật toán đúng

**Robot ở trên viền.** Ban đầu ở góc `(0,0)` (trên viền). Đi theo một hướng dọc theo cạnh viền giữ robot trên cạnh đó; khi tới góc, bước tiếp theo ra ngoài ⇒ quay ngược chiều kim đồng hồ đúng một lần (lúc đó hướng mới chỉ dọc theo cạnh kế tiếp, ô kế tiếp hợp lệ vì `w, h ≥ 2`) ⇒ vẫn trên viền. Quy nạp: mọi bước đi chuyển robot sang ô **kế tiếp trên vòng** theo thứ tự đã đánh số.

**Chỉ số.** Do đó sau `t` bước robot ở ô `ring[t mod P]`.

**Hướng.** Robot chỉ quay khi đang **thử** bước; sau khi dừng, hướng là hướng của bước vừa thực hiện — đúng nhãn đã ghi cho ô đó: các ô cạnh dưới (`k = 1..w−1`) tới bằng bước East; cạnh phải bằng bước North; cạnh trên bằng West; cạnh trái và `(0,0)` (với `t ≥ 1`) bằng South. Nếu chưa đi bước nào, hướng là East ban đầu. ∎

### 10.1 Kiểm chứng thực nghiệm: mọi chỉ số `k` của mọi lưới nhỏ

Với mọi `w, h ≤ 7` và mọi tổng số bước `t ∈ [0, 3P]`, so trạng thái của mô phỏng từng bước với bảng vòng.

In [11]:
cnt = 0
for w in range(2, 8):
    for h in range(2, 8):
        rb, rc = RobotBrute(w, h), Robot(w, h); P = rc.P
        assert rb.getPos() == rc.getPos() and rb.getDir() == rc.getDir()
        for t in range(1, 3 * P + 1):
            rb.step(1); rc.step(1)
            assert rb.getPos() == rc.getPos() and rb.getDir() == rc.getDir(), (w, h, t)
            cnt += 1
print(f"Bảng vòng = mô phỏng từng bước trên {cnt} trạng thái (mọi lưới 2..7 x 2..7, 3 vòng)")

Bảng vòng = mô phỏng từng bước trên 1512 trạng thái (mọi lưới 2..7 x 2..7, 3 vòng)


## 11. Phân tích Time / Space Complexity

| Cách | `__init__` | `step` | `getPos`/`getDir` | Space |
|---|---|---|---|---|
| 1. Từng bước | `O(1)` | `O(num)` | `O(1)` | `O(1)` |
| 2. Bảng vòng | `O(w + h)` | `O(1)` | `O(1)` | `O(w + h)` |
| 3. Công thức | `O(1)` | `O(1)` | `O(1)` | `O(1)` |
| 4. Theo cạnh | `O(1)` | `O(1)` (≤ 5 đoạn) | `O(1)` | `O(1)` |

In [12]:
rng = random.Random(2069)
ops, args = ["Robot"], [[100, 100]]
for _ in range(10 ** 4 - 1):
    op = rng.choice(["step", "getPos", "getDir"]); ops.append(op); args.append([rng.randint(1, 10 ** 5)] if op == "step" else [])
for cls in (Robot, RobotFormula, RobotSegments):
    t = time.perf_counter(); r = run(cls, ops, args); print(f"{cls.__name__:<13}: {(time.perf_counter() - t) * 1000:.0f} ms")
small_ops, small_args = ops[:60], args[:60]
t = time.perf_counter(); run(RobotBrute, small_ops, small_args); print(f"RobotBrute   : 60 lệnh đầu {(time.perf_counter() - t) * 1000:.0f} ms (x166 cho toàn bộ)")

Robot        : 1 ms
RobotFormula : 2 ms
RobotSegments: 3 ms
RobotBrute   : 60 lệnh đầu 74 ms (x166 cho toàn bộ)


## 12. Bộ test

Sinh chuỗi lệnh ngẫu nhiên (seed cố định, có cả `num` là bội của `P`), đối chiếu mọi cài đặt với `RobotBrute`.

In [13]:
IMPLS = [Robot, RobotFormula, RobotSegments]
def check(ops, args):
    ref = run(RobotBrute, ops, args)
    for cls in IMPLS:
        assert run(cls, ops, args) == ref, (cls.__name__, ops, args)
    return ref

def random_ops(rng, q, maxnum=60):
    w, h = rng.randint(2, 7), rng.randint(2, 7); P = 2 * (w - 1) + 2 * (h - 1)
    ops, args = ["Robot"], [[w, h]]
    for _ in range(q):
        op = rng.choice(["step", "getPos", "getDir", "getDir"])
        ops.append(op)
        if op == "step":
            args.append([rng.choice([rng.randint(1, maxnum), P, 2 * P, P - 1, P + 1])])
        else:
            args.append([])
    return ops, args

assert check(ex_ops, ex_args) == ex_out
# chuỗi lệnh tự dựng (đối chiếu lại đề gốc) — kỳ vọng kiểm bằng mô phỏng từng bước trong check()
edge = [
    (["Robot", "getPos", "getDir"], [[2, 2], [], []], [None, [0, 0], "East"]),
    (["Robot", "step", "getPos", "getDir"], [[2, 2], [4], [], []], [None, None, [0, 0], "South"]),
    (["Robot", "step", "getDir", "step", "getDir"], [[2, 3], [1], [], [1], []], [None, None, "East", None, "North"]),
    (["Robot", "step", "getPos", "getDir"], [[3, 2], [3], [], []], [None, None, [2, 1], "North"]),
]
for ops, args, exp in edge:
    assert check(ops, args) == exp, (ops, check(ops, args))
rng = random.Random(20690)
for _ in range(500):
    check(*random_ops(rng, rng.randint(1, 25)))
print("Ví dụ, 4 chuỗi tự dựng, 500 chuỗi lệnh ngẫu nhiên: OK")

Ví dụ, 4 chuỗi tự dựng, 500 chuỗi lệnh ngẫu nhiên: OK


### 12.1 Kiểm tra tính chất

- `step(a)` rồi `step(b)` cho cùng trạng thái với `step(a + b)`.
- Sau khi đã đi, `step(P)` không đổi trạng thái (vị trí lẫn hướng).
- `getPos`/`getDir` không đổi trạng thái (gọi nhiều lần cho cùng kết quả).

In [14]:
rng = random.Random(121)
for _ in range(1000):
    w, h = rng.randint(2, 20), rng.randint(2, 20); a, b = rng.randint(1, 500), rng.randint(1, 500)
    r1, r2 = Robot(w, h), Robot(w, h); r1.step(a); r1.step(b); r2.step(a + b)
    assert (r1.getPos(), r1.getDir()) == (r2.getPos(), r2.getDir())
    st = (r1.getPos(), r1.getDir()); r1.step(r1.P); assert (r1.getPos(), r1.getDir()) == st
    assert r1.getPos() == r1.getPos() and r1.getDir() == r1.getDir()
print("3 tính chất đúng trên 1000 lần thử")

3 tính chất đúng trên 1000 lần thử


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Trả "East" ở `(0, 0)` sau khi đã đi một vòng

In [15]:
class WrongStartDir(Robot):
    def getDir(self):
        return "East" if self.k == 0 else self.cells[self.k][2]             # SAI

ops = ["Robot", "step", "getDir"]; args = [[2, 2], [4], []]                   # tự dựng (đối chiếu lại đề gốc)
print("sai :", run(WrongStartDir, ops, args), "| đúng:", run(Robot, ops, args), "| brute:", run(RobotBrute, ops, args))

sai : [None, None, 'East'] | đúng: [None, None, 'South'] | brute: [None, None, 'South']


**Vì sao sai:** robot tới `(0,0)` bằng bước đi **xuống** cạnh trái ⇒ hướng South; nó chỉ quay sang East khi **thử** bước tiếp theo. **Sửa:** phân biệt "chưa đi" (East) và "đã đi" (South) tại chỉ số 0.

### Lỗi 2 — `num %= P; if num == 0: return` (bỏ qua vòng trọn)

In [16]:
class WrongSkipFullLoop(RobotSegments):
    def step(self, num):
        num %= self.P
        if num == 0:
            return                                                        # SAI: hướng có thể đổi sau một vòng
        RobotSegments.step(self, num)

ops = ["Robot", "step", "getDir", "getPos"]; args = [[3, 3], [8], [], []]      # tự dựng (đối chiếu lại đề gốc): P = 8
print("sai :", run(WrongSkipFullLoop, ops, args), "| đúng:", run(Robot, ops, args), "| brute:", run(RobotBrute, ops, args))

sai : [None, None, 'East', [0, 0]] | đúng: [None, None, 'South', [0, 0]] | brute: [None, None, 'South', [0, 0]]


**Vì sao sai:** đi đủ một vòng từ trạng thái ban đầu đưa robot về `(0,0)` nhưng hướng thành **South**. Bỏ qua lệnh làm hướng vẫn là East. **Sửa:** khi `num % P == 0`, vẫn "đi" một vòng (hoặc dùng cờ `moved`).

### Lỗi 3 — Quay theo chiều kim đồng hồ

In [17]:
class WrongClockwise(RobotBrute):
    def step(self, num):
        for _ in range(num):
            while True:
                nx, ny = self.x + DV[self.d][0], self.y + DV[self.d][1]
                if 0 <= nx < self.w and 0 <= ny < self.h:
                    self.x, self.y = nx, ny; break
                self.d = (self.d - 1) % 4                                 # SAI

print("sai :", run(WrongClockwise, ex_ops, ex_args), "\nđúng:", ex_out)

sai : [None, None, None, [4, 0], 'East', None, None, None, [0, 1], 'North'] 
đúng: [None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']


**Vì sao sai:** từ East quay **ngược** chiều kim đồng hồ là North; quay xuôi là South — robot đi ra ngoài rồi lại quay, quỹ đạo khác hẳn (thậm chí bị kẹt quay ở góc). **Sửa:** `d + 1` theo thứ tự `East, North, West, South`.

### Lỗi 4 — Chu vi `2(w + h)` (đếm góc hai lần)

In [18]:
class WrongPerimeter(RobotFormula):
    def __init__(self, w, h):
        super().__init__(w, h); self.P = 2 * (w + h)                       # SAI

print("sai :", run(WrongPerimeter, ex_ops, ex_args), "\nđúng:", ex_out)
ops = ["Robot", "step", "getPos"]; args = [[6, 3], [15], []]
print("sai :", run(WrongPerimeter, ops, args), "| đúng:", run(Robot, ops, args))

sai : [None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West'] 
đúng: [None, None, None, [4, 0], 'East', None, None, None, [1, 2], 'West']
sai : [None, None, [0, -1]] | đúng: [None, None, [1, 0]]


**Vì sao sai:** số **bước** đi hết một vòng là `2(w − 1) + 2(h − 1)` (số ô trên viền). Ví dụ 1 còn đúng vì chưa đi hết vòng, nhưng `step(15)` lộ lỗi (và công thức vị trí cũng có thể trả tọa độ ngoài lưới). **Sửa:** `P = 2(w − 1) + 2(h − 1)`.

### Lỗi 5 — Mô phỏng từng bước (TLE)

In [19]:
ops = ["Robot"] + ["step"] * 20; args = [[100, 100]] + [[10 ** 5]] * 20
for cls in (RobotBrute, Robot):
    t = time.perf_counter(); run(cls, ops, args); print(f"{cls.__name__:<11}: 20 lệnh step(1e5) {(time.perf_counter() - t) * 1000:.0f} ms")

RobotBrute : 20 lệnh step(1e5) 212 ms
Robot      : 20 lệnh step(1e5) 0 ms


**Vì sao không tốt:** tới `10⁴` lệnh `step(10⁵)` ⇒ `10⁹` bước. **Sửa:** modulo chu vi.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mô phỏng chuyển động có tính **tuần hoàn** ⇒ tìm chu kỳ rồi lấy **modulo**.
- Cẩn thận trạng thái **ban đầu** khác với trạng thái "sau một chu kỳ" (ở đây hướng tại `(0,0)`).
- Hướng khi dừng là hướng của bước **vừa đi**, không phải hướng sẽ quay sang.
- Bài Design: luôn có mô phỏng chậm để đối chiếu ngẫu nhiên.

## 15. Biến thể của bài

1. **874 Walking Robot Simulation** (bản I): lệnh rẽ trái/phải và chướng ngại vật.
2. **1041 Robot Bounded In Circle**: sau 4 lần lặp lệnh, robot về gốc ⇔ bị giới hạn trong vòng tròn.
3. **Robot quay theo chiều kim đồng hồ** và xuất phát ở góc khác — dựng bảng vòng tương ứng.
4. **Lưới có `w = 1` hoặc `h = 1`** (ngoài ràng buộc): vòng suy biến thành đi tới–lui.
5. **Truy vấn "sau bao nhiêu bước robot lần đầu tới ô `(x, y)`"** — tra ngược bảng vòng.

### 15.1 Code biến thể: 1041 Robot Bounded In Circle

In [20]:
def is_robot_bounded(instructions):
    x = y = 0; d = 0                                    # 0 = North
    D4 = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    for ch in instructions:
        if ch == "G": x += D4[d][0]; y += D4[d][1]
        elif ch == "L": d = (d - 1) % 4
        else: d = (d + 1) % 4
    return (x, y) == (0, 0) or d != 0

def bounded_brute(instructions, rounds=8):
    x = y = 0; d = 0; D4 = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    for _ in range(rounds):
        for ch in instructions:
            if ch == "G": x += D4[d][0]; y += D4[d][1]
            elif ch == "L": d = (d - 1) % 4
            else: d = (d + 1) % 4
        if (x, y) == (0, 0) and d == 0: return True
    return False

rng = random.Random(1041)
for _ in range(500):
    s = "".join(rng.choice("GLR") for _ in range(rng.randint(1, 10))); assert is_robot_bounded(s) == bounded_brute(s)
print("1041 khớp mô phỏng 8 vòng;", is_robot_bounded("GGLLGG"), is_robot_bounded("GG"), is_robot_bounded("GL"))

1041 khớp mô phỏng 8 vòng; True False True


## 16. So sánh các cách làm

| Cách | Ý tưởng | `step` | Khi nào dùng |
|---|---|---|---|
| 1. Từng bước | Sát đề | `O(num)` | Kiểm chứng |
| 2. Bảng vòng | Chỉ số mod `P` | `O(1)` | **Bản nộp** |
| 3. Công thức | Tính từ chỉ số | `O(1)` | Không cần bảng |
| 4. Theo cạnh | Mod + nhảy đoạn | `O(1)` | Giữ dạng mô phỏng |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Lưới `2 × 3`: `step(1), getDir, step(1), getDir, step(4), getPos, getDir`.

**Bài 2 (tính tay):** Dựng bảng vòng cho lưới `4 × 3` (chỉ số, ô, hướng) và cho biết trạng thái sau `step(23)`.

**Bài 3 (điền TODO):** Hoàn thiện `perimeter(w, h)`.

**Bài 4 (tự viết lại):** Thêm phương thức `firstVisit(x, y)` trả số bước nhỏ nhất (tính từ đầu) để robot tới ô `(x, y)` trên viền, hoặc `−1` nếu ô không trên viền.

**Bài 5 (mini challenge):** Viết `RobotCW`: robot xuất phát ở `(0, 0)` hướng **North** và quay **theo** chiều kim đồng hồ khi bị chặn. Dựng bảng vòng tương ứng (cẩn thận hướng ở `(0,0)` sau một vòng) và đối chiếu mô phỏng từng bước.

In [21]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def perimeter(w, h):
    # TODO: số ô trên viền = 2 * (w - 1) + 2 * (h - 1)
    pass

In [22]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
class RobotFirstVisit(Robot):
    # TODO: lưu dict ô -> chỉ số trong bảng vòng; (0,0) -> 0
    pass

class RobotCW:
    # TODO: bảng vòng đi lên cạnh trái, sang phải cạnh trên, xuống cạnh phải, sang trái cạnh dưới
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: `P = 2·1 + 2·2 = 6`. Mức 2: `step(1)` ⇒ `(1,0)` East; `step(1)` ⇒ `(1,1)` North. Mức 3: `step(4)` ⇒ tổng 6 ⇒ `(0,0)` South.

**Bài 2.** Mức 1: `P = 2·3 + 2·2 = 10`. Mức 2: cạnh dưới `k = 0..3`, phải `4..5`, trên `6..8`, trái `9`. Mức 3: `23 mod 10 = 3` ⇒ `(3, 0)` East.

**Bài 3.** Mức 1: hai cạnh ngang `w − 1` bước mỗi cạnh. Mức 2: hai cạnh dọc `h − 1`. Mức 3: cộng lại.

**Bài 4.** Mức 1: ô trên viền ⇔ `x ∈ {0, w−1}` hoặc `y ∈ {0, h−1}`. Mức 2: tra chỉ số trong bảng vòng. Mức 3: `(0,0)` là bước 0.

**Bài 5.** Mức 1: thứ tự hướng xuôi chiều kim đồng hồ `North, East, South, West`. Mức 2: bảng vòng: lên cạnh trái, … Mức 3: tại `(0,0)` sau một vòng robot tới bằng bước **West** (đi dọc cạnh dưới sang trái).

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1, 2, 3
b1 = run(Robot, ["Robot", "step", "getDir", "step", "getDir", "step", "getPos", "getDir"], [[2, 3], [1], [], [1], [], [4], [], []])
print("Bài 1:", b1); assert b1 == [None, None, "East", None, "North", None, [0, 0], "South"] == run(RobotBrute, ["Robot", "step", "getDir", "step", "getDir", "step", "getPos", "getDir"], [[2, 3], [1], [], [1], [], [4], [], []])
rb = Robot(4, 3); print("Bài 2: bảng vòng", [(i, c) for i, c in enumerate(rb.cells)]); rb.step(23); print("   sau step(23):", rb.getPos(), rb.getDir())
assert (rb.getPos(), rb.getDir()) == ([3, 0], "East")
def perimeter(w, h):
    return 2 * (w - 1) + 2 * (h - 1)
assert all(perimeter(w, h) == Robot(w, h).P for w in range(2, 10) for h in range(2, 10))
print("Bài 3: OK")

Bài 1: [None, None, 'East', None, 'North', None, [0, 0], 'South']
Bài 2: bảng vòng [(0, (0, 0, 'South')), (1, (1, 0, 'East')), (2, (2, 0, 'East')), (3, (3, 0, 'East')), (4, (3, 1, 'North')), (5, (3, 2, 'North')), (6, (2, 2, 'West')), (7, (1, 2, 'West')), (8, (0, 2, 'West')), (9, (0, 1, 'South'))]
   sau step(23): [3, 0] East
Bài 3: OK


In [24]:
# Bài 4, 5
class RobotFirstVisit(Robot):
    def __init__(self, w, h):
        super().__init__(w, h); self.index = {}
        for i, (x, y, _) in enumerate(self.cells):
            self.index.setdefault((x, y), i)
    def firstVisit(self, x, y):
        return self.index.get((x, y), -1)

class RobotCW:
    NAMES = ["North", "East", "South", "West"]; DV = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    def __init__(self, w, h):
        cells = []
        for y in range(h): cells.append((0, y, "North"))
        for x in range(1, w): cells.append((x, h - 1, "East"))
        for y in range(h - 2, -1, -1): cells.append((w - 1, y, "South"))
        for x in range(w - 2, 0, -1): cells.append((x, 0, "West"))
        cells[0] = (0, 0, "West")
        self.cells, self.P, self.k, self.moved = cells, len(cells), 0, False
    def step(self, num): self.moved = True; self.k = (self.k + num) % self.P
    def getPos(self): return list(self.cells[self.k][:2])
    def getDir(self): return self.cells[self.k][2] if self.moved else "North"

class RobotCWBrute:
    def __init__(self, w, h): self.w, self.h, self.x, self.y, self.d = w, h, 0, 0, 0
    def step(self, num):
        for _ in range(num):
            while True:
                nx, ny = self.x + RobotCW.DV[self.d][0], self.y + RobotCW.DV[self.d][1]
                if 0 <= nx < self.w and 0 <= ny < self.h: self.x, self.y = nx, ny; break
                self.d = (self.d + 1) % 4
    def getPos(self): return [self.x, self.y]
    def getDir(self): return RobotCW.NAMES[self.d]

rng = random.Random(5)
for _ in range(300):
    w, h = rng.randint(2, 7), rng.randint(2, 7); fv = RobotFirstVisit(w, h); rb = RobotBrute(w, h); seen = {(0, 0): 0}
    for t in range(1, fv.P + 1):
        rb.step(1); seen.setdefault(tuple(rb.getPos()), t)
    assert all(fv.firstVisit(x, y) == seen.get((x, y), -1) for x in range(w) for y in range(h))
    a, b = RobotCW(w, h), RobotCWBrute(w, h)
    for _ in range(10):
        k = rng.choice([rng.randint(1, 30), a.P]); a.step(k); b.step(k)
        assert (a.getPos(), a.getDir()) == (b.getPos(), b.getDir())
print("Bài 4, 5: OK |", RobotFirstVisit(6, 3).firstVisit(1, 2), RobotFirstVisit(6, 3).firstVisit(2, 1))

Bài 4, 5: OK | 11 -1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template:**

```python
cells = [(x,0,"East") for x in 0..w-1] + [(w-1,y,"North") for y in 1..h-1]
      + [(x,h-1,"West") for x in w-2..0] + [(0,y,"South") for y in h-2..1]
cells[0] = (0, 0, "South")                 # sau khi đã đi
step(num): moved = True; k = (k + num) % P
getPos: cells[k][:2]
getDir: "East" nếu chưa moved, ngược lại cells[k][2]
```

**Checklist:**
- [ ] Chu vi `P = 2(w−1) + 2(h−1)`?
- [ ] Hướng tại `(0,0)`: East khi chưa đi, South khi đã đi?
- [ ] Không bỏ qua lệnh khi `num % P == 0`?
- [ ] Quay **ngược** chiều kim đồng hồ: East → North → West → South?
- [ ] `step` `O(1)` (không mô phỏng từng bước)?